In [1]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
# load model
MODEL_PATH = "/home/et2842/causal/causalrl/models/pointmaze_medium_expert.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Rebuild the model with the same architecture
action_bounds = (checkpoint['action_bounds_low'], checkpoint['action_bounds_high'])

pretrained_actor = ContinuousPolicyNN(
    input_dim=checkpoint['input_dim'],
    action_dim=checkpoint['num_actions'],
    hidden_dim=checkpoint['hidden_dim'],
    num_blocks=checkpoint['num_blocks'],
    dropout=checkpoint['dropout'],
    layernorm=checkpoint['layernorm'],
    final_tanh=checkpoint['final_tanh'],
    action_bounds=action_bounds,
).to(device)

pretrained_actor.load_state_dict(checkpoint['state_dict'])
# pretrained_actor.eval()
pretrained_actor.train()

slots = checkpoint['slots']
Z_trim = checkpoint['Z_trim']
dims = checkpoint['dims']
lookback = checkpoint['lookback']

state_dim = checkpoint['input_dim']
state_dim

/tmp/ipykernel_1438853/2066741687.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location=device)


9

In [4]:
num_steps = 1000
rl_seed_pretrain = 2014
rl_seed = 90210
hidden_dims = set() # {'W'}

env_pretrain = PointMazePCH(num_steps=num_steps, expert_mode=True, seed=rl_seed_pretrain)
env_train = PointMazePCH(num_steps=num_steps, expert_mode=True, seed=rl_seed)
action_dim = env_train.env.action_space.shape[0]
action_dim

2

In [5]:
# reward shaping
def make_dense_distance_reward(env, use_delta=True, c=1.0, success_bonus=50.0, success_radius=5.0):
        goal_xy = env.env._goal_xy
    
        def reward_fn(obs, reward_env):
            t = len(obs['P']) - 1
    
            P_curr = obs['P'][t]
            curr_xy = np.array(P_curr[:2], dtype=np.float64)
            dist_curr = np.linalg.norm(curr_xy - goal_xy)
    
            r = 0.0
            if use_delta:
                if t == 0:
                    r = 0.0
                else:
                    P_prev = obs['P'][t - 1]
                    prev_xy = np.array(P_prev[:2], dtype=np.float64)
                    dist_prev = np.linalg.norm(prev_xy - goal_xy)
                    r = float(c * (dist_prev - dist_curr))
            else:
                r = float(-c * dist_curr)

            if dist_curr <= success_radius:
                r += success_bonus
    
            return r
    
        return reward_fn

reward_fn = make_dense_distance_reward(env_train)

In [6]:
config = OnlineRLConfig(
    # total_env_steps=1_000_000,
    total_env_steps=100_000,
    start_steps=10_000,
    max_episode_steps=num_steps,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    policy_delay=2,
    actor_lr=3e-6,
    critic_lr=3e-4,
    noise_std=0.15,
    hidden_dim_q=256,
    target_policy_noise=0.2,
    target_noise_clip=0.3,
    actor_warmup_steps=50_000,
    bc_reg_lambda=0.01,
    max_grad_norm=1.0
)

In [7]:
# pretrain critics offline
replay_buffer, q1, q2, target_q1, target_q2 = pretrain_critics_offline(
    env=env_pretrain,
    pretrained_actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    num_pretrain_steps=300_000,
    pretrain_updates=150_000,
    seed=rl_seed_pretrain,
    reward_shaping_fn=reward_fn
)

In [8]:
def callback(stats: dict):
    if stats['episode'] % 1 == 0:
        print(
            f'[Episode {stats["episode"]}] '
            f'steps={stats["env_steps"]}, '
            f'return={stats["return"]:.2f}, '
            f'len={stats["length"]}, '
            f'buffer={stats["buffer_size"]}'
        )

In [9]:
fine_tuned_policy, logs = td3_fine_tune_actor(
    env=env_train,
    actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    seed=rl_seed,
    log_callback=callback,
    replay_buffer=replay_buffer,
    initial_q1=q1,
    initial_q2=q2,
    initial_target_q1=target_q1,
    initial_target_q2=target_q2,
    reward_shaping_fn=reward_fn
)

ft_pi = shared_policy_fn_long_horizon(fine_tuned_policy, slots, Z_trim, continuous=True, device=device)
ft_policies = make_shared_policy_dict(ft_pi)

[Episode 1] steps=1000, return=8.11, len=1000, buffer=301000
[Episode 2] steps=2000, return=8.21, len=1000, buffer=302000
[Episode 3] steps=3000, return=8.80, len=1000, buffer=303000
[Episode 4] steps=4000, return=-0.27, len=1000, buffer=304000
[Episode 5] steps=5000, return=0.03, len=1000, buffer=305000
[Episode 6] steps=6000, return=16.78, len=1000, buffer=306000
[Episode 7] steps=7000, return=8.72, len=1000, buffer=307000
[Episode 8] steps=8000, return=7.94, len=1000, buffer=308000
[Episode 9] steps=9000, return=0.36, len=1000, buffer=309000
[Episode 10] steps=10000, return=-0.03, len=1000, buffer=310000
[Episode 11] steps=10176, return=123.44, len=176, buffer=310176
[Episode 12] steps=11176, return=8.35, len=1000, buffer=311176
[Episode 13] steps=12176, return=7.50, len=1000, buffer=312176
[Episode 14] steps=13176, return=-0.10, len=1000, buffer=313176
[Episode 15] steps=14176, return=16.76, len=1000, buffer=314176
[Episode 16] steps=15176, return=7.88, len=1000, buffer=315176
[Epi

In [10]:
expert_env = PointMazePCH(num_steps=num_steps, expert_mode=True)

In [15]:
num_eval_eps = 20

records = collect_imitator_trajectories(
    env=expert_env,
    policies=ft_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

Starting episode 1/20...
  Episode 1 ended at step 106 (terminated: True, truncated: False).
Starting episode 2/20...
  Episode 2 ended at step 1000 (terminated: False, truncated: True).
Starting episode 3/20...
  Episode 3 ended at step 106 (terminated: True, truncated: False).
Starting episode 4/20...
  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/20...
  Episode 5 ended at step 1000 (terminated: False, truncated: True).
Starting episode 6/20...
  Episode 6 ended at step 1000 (terminated: False, truncated: True).
Starting episode 7/20...
  Episode 7 ended at step 108 (terminated: True, truncated: False).
Starting episode 8/20...
  Episode 8 ended at step 106 (terminated: True, truncated: False).
Starting episode 9/20...
  Episode 9 ended at step 107 (terminated: True, truncated: False).
Starting episode 10/20...
  Episode 10 ended at step 1000 (terminated: False, truncated: True).
Starting episode 11/20...
  Episode 11 ended at step 1000 (term

In [12]:
# save expert
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'pointmaze_medium_expert_finetuned.pt')

checkpoint = {
    "state_dict": fine_tuned_policy.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env_train.action_space.shape[0],
    "hidden_dim": config.hidden_dim_q,
    "num_blocks": checkpoint['num_blocks'],
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env_train.action_space.low,
    "action_bounds_high": env_train.action_space.high,
    "input_dim": int(fine_tuned_policy.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

Saved expert to: /home/et2842/causal/causalrl/models/pointmaze_medium_expert_finetuned.pt
